# Chapter 7 - Lab 1: Sequential pipeline with LlamaIndex AgentWorkflow (customer-care edition)

**Goal**: build a 4-agent **sequential pipeline** that assesses the health of a NovaMart **B2B service account**
(e.g. a hotel chain that buys ACs and TVs plus a service contract). The agents (AccountData, ServiceQuality,
RelationshipRisk, Supervisor) pass control through **explicit handoffs** and share data via a `Context` object.

*Finance edition*: a financial-health analyzer (Fundamental -> Profitability -> Liquidity -> Supervisor).
*Customer-care edition*: an **account-health analyzer**:

| Finance agent | Customer-care agent | Metrics |
|---|---|---|
| FundamentalAgent | AccountDataAgent | loads all account metrics |
| ProfitabilityAgent | ServiceQualityAgent | CSAT, first-contact resolution, SLA compliance, self-service |
| LiquidityAgent | RelationshipRiskAgent | repeat contacts, escalations, first response, renewal probability |
| SupervisorAgent | SupervisorAgent | overall account-health verdict |

Labs 2 and 3 solve related tasks with the OpenAI Agents SDK and AutoGen, so you can compare three architectural styles.

**Tech stack**: LlamaIndex `FunctionAgent`, `AgentWorkflow`, `Context` (shared state); OpenAI `gpt-4o`; stub account data in `common.py`.

## 1. Install packages

In [ ]:
%pip install -q llama-index llama-index-llms-openai pydantic python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

## 3. Bootstrap the shared models and helpers

In [ ]:
import os, urllib.request

# If you cloned the repository, the helper files already sit next to this notebook.
# In Colab, either upload them with the Files panel or set RAW_BASE to the raw URL
# of the "customer-care-ai-agents" folder in your copy of the repository.
RAW_BASE = os.environ.get("CC_RAW_BASE", "")   # e.g. https://raw.githubusercontent.com/<you>/<repo>/main/customer-care-ai-agents
for name in ['common.py']:
    if not os.path.exists(name):
        if not RAW_BASE:
            raise FileNotFoundError(f"{name} not found: upload it next to the notebook or set CC_RAW_BASE")
        urllib.request.urlretrieve(f"{RAW_BASE}/Chapter%207/{name}", name)
print("Helper files ready:", ['common.py'])

In [ ]:
from common import (
    SERVICE_QUALITY_THRESHOLDS, RELATIONSHIP_RISK_THRESHOLDS,
    format_threshold_prompt, stub_lookup,
)
print("Thresholds and stub data loaded.")

## 4. Define the four agent tools

Each agent has a dedicated tool that reads from and writes to a shared `Context`: the **shared-state
communication pattern**. Instead of passing data through return values, agents collaborate via one
state object stored in `ctx.store`.

In [ ]:
from llama_index.core.workflow import Context


async def get_account_metrics(ctx: Context, account_id: str) -> str:
    """Used by AccountDataAgent. Reads the account's metrics into shared state."""
    data = stub_lookup(account_id)
    state = await ctx.store.get("state")
    if state["account_id"] == "":
        state["account_id"] = account_id
    state["metrics"] = {**data["service_quality"], **data["relationship_risk"]}
    await ctx.store.set("state", state)
    return f"Metrics extracted for {account_id}."


async def get_service_quality_metrics(ctx: Context) -> str:
    """Used by ServiceQualityAgent. Selects the service-quality metrics and their thresholds."""
    state = await ctx.store.get("state")
    if not state["metrics"]:
        return "No metrics - run get_account_metrics first."
    state["service_quality_metrics"] = {k: state["metrics"][k] for k in SERVICE_QUALITY_THRESHOLDS}
    state["threshold_service_quality_comments"] = format_threshold_prompt(
        SERVICE_QUALITY_THRESHOLDS, "Service Quality")
    await ctx.store.set("state", state)
    return (f"Service-quality metrics for {state['account_id']}: {state['service_quality_metrics']}\n"
            f"{state['threshold_service_quality_comments']}")


async def get_relationship_risk_metrics(ctx: Context) -> str:
    """Used by RelationshipRiskAgent. Selects the relationship-risk metrics and their thresholds."""
    state = await ctx.store.get("state")
    if not state["metrics"]:
        return "No metrics - run get_account_metrics first."
    state["relationship_risk_metrics"] = {k: state["metrics"][k] for k in RELATIONSHIP_RISK_THRESHOLDS}
    state["threshold_relationship_risk_comments"] = format_threshold_prompt(
        RELATIONSHIP_RISK_THRESHOLDS, "Relationship Risk")
    await ctx.store.set("state", state)
    return (f"Relationship-risk metrics for {state['account_id']}: {state['relationship_risk_metrics']}\n"
            f"{state['threshold_relationship_risk_comments']}")


async def get_overall_comments(ctx: Context) -> str:
    """Used by SupervisorAgent. Confirms both scorings are done before the final verdict."""
    state = await ctx.store.get("state")
    if state["threshold_service_quality_comments"] is None:
        return "Service-quality scoring not done."
    if state["threshold_relationship_risk_comments"] is None:
        return "Relationship-risk scoring not done."
    state["overall_comments"] = "Synthesis complete."
    await ctx.store.set("state", state)
    return "Both scorings are complete; write the overall account-health verdict."

## 5. Define the four agents and their handoffs

Each agent has a focused prompt, one tool, and an explicit `can_handoff_to` naming the **next** agent.
Two disciplines keep the run deterministic: every prompt says *call your tool exactly once*, and the
Supervisor (last agent) writes the final answer instead of handing off.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-4o", temperature=0)

account_data_agent = FunctionAgent(
    name="AccountDataAgent",
    description="Collect the service metrics of a B2B account.",
    system_prompt=("You are the account-data analyst. Call get_account_metrics exactly once for the "
                   "requested account, then hand off to the ServiceQualityAgent. Never call the tool twice."),
    llm=llm,
    tools=[get_account_metrics],
    can_handoff_to=["ServiceQualityAgent"],
)

service_quality_agent = FunctionAgent(
    name="ServiceQualityAgent",
    description="Score service-quality metrics against thresholds.",
    system_prompt=("You are the ServiceQualityAgent. Call get_service_quality_metrics exactly once, then score "
                   "each service-quality metric from 1 to 10 against the thresholds, with a data-backed "
                   "justification for each. Then hand off to the RelationshipRiskAgent."),
    llm=llm,
    tools=[get_service_quality_metrics],
    can_handoff_to=["RelationshipRiskAgent"],
)

relationship_risk_agent = FunctionAgent(
    name="RelationshipRiskAgent",
    description="Score relationship-risk metrics against thresholds.",
    system_prompt=("You are the RelationshipRiskAgent. Call get_relationship_risk_metrics exactly once, then "
                   "score each relationship-risk metric from 1 to 10 against the thresholds (for 'lower is "
                   "better' metrics, a low value earns a high score), with a data-backed justification for each. "
                   "Then hand off to the SupervisorAgent."),
    llm=llm,
    tools=[get_relationship_risk_metrics],
    can_handoff_to=["SupervisorAgent"],
)

# can_handoff_to=[] makes the Supervisor terminal: omitting the argument would mean
# "can hand off to ANY agent" in AgentWorkflow.
supervisor_agent = FunctionAgent(
    name="SupervisorAgent",
    description="Synthesise the overall account-health verdict.",
    system_prompt=("You are the Supervisor (head of customer success). Call get_overall_comments exactly once, "
                   "then write the final account-health report yourself: list every service-quality and "
                   "relationship-risk metric with its 1-10 score and a one-line justification, give an overall "
                   "verdict (HEALTHY | WATCH | AT RISK | CRITICAL) and the top two actions for the account team. "
                   "You are the last agent in the pipeline: do not hand off."),
    llm=llm,
    tools=[get_overall_comments],
    can_handoff_to=[],
)

## 6. Assemble the workflow

In [ ]:
from llama_index.core.agent.workflow import AgentWorkflow

agent_workflow = AgentWorkflow(
    agents=[account_data_agent, service_quality_agent, relationship_risk_agent, supervisor_agent],
    root_agent=account_data_agent.name,
    initial_state={
        "metrics": {},
        "account_id": "",
        "service_quality_metrics": {},
        "relationship_risk_metrics": {},
        "threshold_service_quality_comments": None,
        "threshold_relationship_risk_comments": None,
        "overall_comments": None,
    },
)

## 7. Run the workflow and observe agent transitions

LlamaIndex's default budget of 20 workflow iterations is too tight for four agents (each spends about three iterations: tool call, handoff, response), so we raise it to `max_iterations=60`.

In [ ]:
from llama_index.core.agent.workflow import AgentStream, ToolCallResult

account_id = "AC-HOTELS"
handler = agent_workflow.run(
    user_msg=f"Assess the account health of {account_id} and comment on the relationship.",
    max_iterations=60,
)

last_agent = None
async for event in handler.stream_events():
    if isinstance(event, ToolCallResult):
        print(f"\n[{event.tool_name}] -> {str(event.tool_output)[:90]}")
    elif isinstance(event, AgentStream) and event.delta:
        if event.current_agent_name != last_agent:
            print(f"\n{'=' * 40}\n  Active agent: {event.current_agent_name}\n{'=' * 40}")
            last_agent = event.current_agent_name
        print(event.delta, end="", flush=True)

result = await handler
print("\n\nFinal result:\n" + str(result))

## 8. Results

Sunrise Hotels (`AC-HOTELS`) should come out **AT RISK**: CSAT 78% (below the 85% healthy line), SLA compliance
88% (below 90%), escalations 12%, first response 9.5 h and renewal probability 55%.

**What to notice about the sequential-pipeline style**
* **Predictability**: the handoff graph is declared up front; easy to audit.
* **Specialisation**: narrow prompts per agent usually beat one generalist prompt.
* **Shared state vs message passing**: Lab 2 passes control via handoff messages; Lab 3 via raw conversation.
* **Termination discipline**: the last agent writes the answer instead of handing off.
* **Trade-off**: no parallelism within one account; Lab 5 parallelises *across* accounts.

## Discussion
Try `AC-SCHOOLS` (healthy) and `AC-RETAIL` (critical). Which single metric would you show a COO?